In [9]:
# use first lat, last lat, I should really have flights use the last lat, lon in the ADS-B sequence where it thinks there was a flight. 
# Also first lat, last lat has to be on American FAA airspace. 

In [10]:
from datetime import date

from data_engineering.flights.flight_type import get_flights
from flights.evaluation.american_evals import continental_usa_to_continental_usa
import polars as pl

target_date = date(2026,3,15)
df_flights = get_flights(target_date)
df_flights = df_flights.filter(pl.col("ladd") | pl.col("pia"))

all_flight_counts_by_icao = df_flights.group_by("icao").agg(pl.len().alias("all_flight_count"))
continental_flight_counts_by_icao = (
    continental_usa_to_continental_usa(df_flights)
    .group_by("icao")
    .agg(pl.len().alias("continental_flight_count"))
)
continental_only_icaos = (
    all_flight_counts_by_icao
    .join(continental_flight_counts_by_icao, on="icao", how="left")
    .filter(pl.col("all_flight_count") == pl.col("continental_flight_count"))
    .select("icao")
)

df_flights = df_flights.join(continental_only_icaos, on="icao", how="inner")

In [11]:
TELEPORT_MIN_DISTANCE_KM = 50.0
EARTH_RADIUS_KM = 6371.0088

prev_lat = pl.col("prev_last_lat").radians()
prev_lon = pl.col("prev_last_lon").radians()
curr_lat = pl.col("first_lat").radians()
curr_lon = pl.col("first_lon").radians()
haversine_a = (
    ((curr_lat - prev_lat) / 2).sin().pow(2)
    + prev_lat.cos() * curr_lat.cos() * ((curr_lon - prev_lon) / 2).sin().pow(2)
)

teleport_flights = (
    df_flights
    .sort(["icao", "takeoff_time", "landing_time"])
    .with_columns(
        pl.col("landing_time").shift(1).over("icao").alias("prev_landing_time"),
        pl.col("landing_airport_ident").shift(1).over("icao").alias("prev_landing_airport_ident"),
        pl.col("last_lat").shift(1).over("icao").alias("prev_last_lat"),
        pl.col("last_lon").shift(1).over("icao").alias("prev_last_lon"),
    )
    .filter(
        pl.all_horizontal(
            pl.col("prev_last_lat").is_not_null(),
            pl.col("prev_last_lon").is_not_null(),
            pl.col("first_lat").is_not_null(),
            pl.col("first_lon").is_not_null(),
        )
    )
    .filter(
        pl.col("prev_landing_airport_ident").is_null()
        | pl.col("takeoff_airport_ident").is_null()
        | (pl.col("prev_landing_airport_ident") != pl.col("takeoff_airport_ident"))
    )
    .with_columns(
        (2 * EARTH_RADIUS_KM * haversine_a.sqrt().clip(0, 1).arcsin()).alias("endpoint_jump_km"),
        ((pl.col("takeoff_time") - pl.col("prev_landing_time")).dt.total_seconds() / 3600).alias("time_gap_hours"),
    )
    .with_columns(
        (pl.col("endpoint_jump_km") / pl.col("time_gap_hours")).alias("required_ground_speed_kmh")
    )
    .filter(pl.col("endpoint_jump_km") >= TELEPORT_MIN_DISTANCE_KM)
    .select(
        "icao",
        "prev_landing_time",
        "takeoff_time",
        "time_gap_hours",
        "endpoint_jump_km",
        "required_ground_speed_kmh",
        "prev_landing_airport_ident",
        "takeoff_airport_ident",
        "prev_last_lat",
        "prev_last_lon",
        "first_lat",
        "first_lon",
        "pia",
        "ladd",
        "aircraft_type",
        "owner",
    )
    .sort("endpoint_jump_km", descending=True)
)

teleport_icaos = (
    teleport_flights
    .group_by("icao")
    .agg(
        pl.len().alias("teleport_count"),
        pl.max("endpoint_jump_km").round(1).alias("max_endpoint_jump_km"),
        pl.min("time_gap_hours").round(2).alias("min_time_gap_hours"),
        pl.col("takeoff_time").min().alias("first_teleport_takeoff_time"),
    )
    .sort("max_endpoint_jump_km", descending=True)
)

teleport_icaos

icao,teleport_count,max_endpoint_jump_km,min_time_gap_hours,first_teleport_takeoff_time
str,u32,f64,f64,datetime[ms]
"""ada9bd""",1,314.0,5.36,2026-03-15 17:46:13.895
"""ad0e7f""",1,238.8,6.34,2026-03-15 12:46:16.990
"""ac0da0""",1,207.5,7.69,2026-03-15 17:00:16.780
"""a7b8ad""",1,189.4,9.14,2026-03-15 14:55:55.905
"""a16891""",1,174.7,5.17,2026-03-15 23:44:43.610
…,…,…,…,…
"""a23195""",1,55.5,6.06,2026-03-15 15:21:15.860
"""a211f4""",1,53.8,1.11,2026-03-15 16:08:17.880
"""a6e057""",1,53.1,3.88,2026-03-15 21:44:24.300


In [8]:
df_flights.filter(pl.col("icao") == "a16891")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,first_message_time,first_lat,first_lon,first_baro_altitude_ft,first_geom_altitude_ft,last_message_time,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,datetime[ms],f64,f64,i64,i64,datetime[ms],f64,f64,i64,i64,bool,bool,bool,bool,str,str,str,str
"""a16891""","""""","""""",2026-03-15 13:56:39.240,"""KHTW""",2026-03-15 14:31:39.670,"""KHAO""",2026-03-15 14:04:08.700,38.419601,-82.50201,6000,6050,2026-03-15 14:31:39.670,39.364162,-84.52336,null,null,false,true,false,false,"""BE20""","""KAR KING LLC""","""BEECH 200 Super King Air""","""A1"""
"""a16891""","""""","""""",2026-03-15 18:07:29.210,"""KHAO""",2026-03-15 18:34:37.450,"""KDWU""",2026-03-15 18:07:27.820,39.360615,-84.509216,1200,null,2026-03-15 18:34:37.450,38.472198,-82.75177,4700,4550,false,true,false,false,"""BE20""","""KAR KING LLC""","""BEECH 200 Super King Air""","""A1"""
"""a16891""","""""","""""",2026-03-15 23:44:43.610,"""KLNP""",2026-03-16 00:22:45.950,"""KTYS""",2026-03-15 23:55:12.120,36.903076,-82.663402,8275,8225,2026-03-16 00:22:45.950,35.832654,-83.964591,1650,1375,false,true,false,false,"""BE20""","""KAR KING LLC""","""BEECH 200 Super King Air""","""A1"""


In [4]:
df_flights.filter(pl.col("icao") == "ada9bd")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,first_message_time,first_lat,first_lon,first_baro_altitude_ft,first_geom_altitude_ft,last_message_time,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,datetime[ms],f64,f64,i64,i64,datetime[ms],f64,f64,i64,i64,bool,bool,bool,bool,str,str,str,str
"""ada9bd""","""""","""""",2026-03-15 11:17:04.820,"""85V""",2026-03-15 12:24:33.470,"""KPHX""",2026-03-15 11:39:18.770,35.719254,-109.93,18050,18475,2026-03-15 12:26:34.110,33.425903,-112.020845,null,null,false,true,false,false,"""BE9L""","""GUARDIAN FLIGHT LLC""","""BEECH 90 King Air""","""A1"""
"""ada9bd""","""""","""""",2026-03-15 17:46:13.895,"""KP10""",2026-03-15 18:53:15.620,"""KDVT""",2026-03-15 18:11:14.910,35.724289,-110.027618,20075,20650,2026-03-15 18:53:15.620,33.686783,-112.083164,null,null,false,true,false,false,"""BE9L""","""GUARDIAN FLIGHT LLC""","""BEECH 90 King Air""","""A1"""


In [ ]:
from data_engineering.adsb.read_adsb import read_adsb
from flights.evaluation.visual import visualize_flight


visualize_flight(df_flights.filter(pl.col("icao") == "a16891"), read_adsb(target_date, icaos=["a16891"]))